In [1]:
# ============================================================
# PIPELINE RAG COMPLETO (CARREGAMENTO, EMBEDDINGS, RECUPERAÇÃO, GERAÇÃO)
# ============================================================

import sys
import os

In [2]:
# Adiciona a raiz do projeto ao sys.path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
print("✅ Caminho adicionado:", sys.path[0])

# ============================================================
# 1. Importações
# ============================================================
from src.rag_pipeline import RAGPipeline
import pandas as pd
import os as os_env

✅ Caminho adicionado: d:\3.1.pos_e_graduacao\INFNET\modulos\sistemas_cognitivos\sistemas_cognitivos_com_llms\pd-sistemas-cognitivos


d:\3.1.pos_e_graduacao\INFNET\modulos\sistemas_cognitivos\sistemas_cognitivos_com_llms\pd-sistemas-cognitivos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 2. Criar Corpus de Exemplo (Artigos Científicos)
# ============================================================
print("\n" + "="*50)
print("CRIANDO CORPUS DE EXEMPLO")
print("="*50)

data = {
    'title': [
        'Introdução à Inteligência Artificial',
        'Deep Learning em Imagens Médicas',
        'Processamento de Linguagem Natural com Transformers',
        'Aprendizado por Reforço em Robótica',
        'Ética e Viés em Sistemas de IA'
    ],
    'abstract': [
        'A inteligência artificial é um campo da ciência da computação que visa criar sistemas capazes de executar tarefas que normalmente requerem inteligência humana, como visão, fala e tomada de decisão.',
        'Deep learning tem revolucionado a análise de imagens médicas. Redes neurais convolucionais alcançam resultados de ponta na detecção de tumores, doenças retinianas e condições patológicas, melhorando o diagnóstico precoce.',
        'Transformers tornaram-se a base dos sistemas modernos de processamento de linguagem natural. Modelos como BERT e GPT utilizam mecanismos de atenção para entender contexto e gerar texto coerente.',
        'Aprendizado por reforço é uma área da IA onde agentes aprendem a tomar decisões através de interação com o ambiente. É amplamente utilizado em robótica, jogos e sistemas autônomos.',
        'Questões éticas como viés algorítmico, privacidade de dados e transparência são desafios importantes no desenvolvimento de IA responsável. A falta de diversidade nos dados de treino pode levar a decisões discriminatórias.'
    ],
    'source': ['arxiv', 'pubmed', 'arxiv', 'arxiv', 'openalex'],
    'year': [2023, 2024, 2023, 2025, 2024]
}

df = pd.DataFrame(data)
df.to_csv("corpus_exemplo.csv", index=False)
print(f"✅ Corpus criado com {len(df)} artigos.")

# ============================================================
# 3. Configurar Pipeline RAG
# ============================================================
print("\n" + "="*50)
print("CONFIGURANDO PIPELINE RAG")
print("="*50)

GROQ_API_KEY = os_env.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    print("⚠️ Chave da API Groq não encontrada.")
    GROQ_API_KEY = input("Digite sua chave da API Groq: ")

pipeline = RAGPipeline(
    corpus_path="corpus_exemplo.csv",
    text_column="abstract",
    groq_api_key=GROQ_API_KEY,
    api_model="llama-3.1-8b-instant",
    k_retrieval=5
)

print("✅ Pipeline carregado com sucesso!")

# ============================================================
# 4. Testar Perguntas
# ============================================================
print("\n" + "="*50)
print("TESTANDO PERGUNTAS")
print("="*50)

perguntas = [
    "Quais são as aplicações da IA mencionadas?",
    "O que é aprendizado por reforço?",
    "Quais são os desafios éticos da IA?",
    "Qual o artigo mais recente do corpus?"
]

for pergunta in perguntas:
    print(f"\n🔍 Pergunta: {pergunta}")
    print("-" * 40)
    resultado = pipeline.answer(pergunta)
    print(f"📝 Resposta:\n{resultado['answer']}\n")
    print(f"📚 Trechos recuperados: {len(resultado['retrieved_context'])}")
    for i, ctx in enumerate(resultado['retrieved_context'][:2]):
        print(f"   {i+1}. {ctx['chunk'][:100]}...")

# ============================================================
# 5. Comparação: Com RAG vs Sem RAG
# ============================================================
print("\n" + "="*50)
print("COMPARAÇÃO: COM RAG vs SEM RAG")
print("="*50)

pergunta_teste = "Quais são os desafios éticos da IA?"

print(f"\n🔍 Pergunta: {pergunta_teste}\n")

# Com RAG (contexto)
print("✅ COM RAG (com contexto):")
resultado_rag = pipeline.answer(pergunta_teste)
print(resultado_rag['answer'])

# Sem RAG (sem contexto)
print("\n❌ SEM RAG (sem contexto):")
resultado_no_rag = pipeline.answer(pergunta_teste, include_context=False)
print(resultado_no_rag['answer'])

print("\n📌 Análise:")
print("   - Com RAG, a resposta é mais específica e fundamentada no corpus.")
print("   - Sem RAG, o modelo tende a gerar respostas genéricas ou alucinadas.")
print("   - O RAG melhora a precisão e reduz alucinações.")

# ============================================================
# 6. Limpeza
# ============================================================
import os as os_clean
if os_clean.path.exists("corpus_exemplo.csv"):
    os_clean.remove("corpus_exemplo.csv")
    print("\n🗑️ Arquivo temporário removido.")

print("\n✅ Notebook executado com sucesso!")


CRIANDO CORPUS DE EXEMPLO
✅ Corpus criado com 5 artigos.

CONFIGURANDO PIPELINE RAG
[1/4] Carregando corpus...
   Anos disponíveis: [np.int64(2023), np.int64(2024), np.int64(2025)]
[2/4] Preparando chunks...
[3/4] Carregando modelo de embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.20it/s]


[4/4] Pipeline RAG pronto (usando Groq)!
✅ Pipeline RAG pronto!
✅ Pipeline carregado com sucesso!

TESTANDO PERGUNTAS

🔍 Pergunta: Quais são as aplicações da IA mencionadas?
----------------------------------------

[LOG] Nova pergunta: Quais são as aplicações da IA mencionadas?
[LOG] Chunk 1 (score: 0.5110): Questões éticas como viés algorítmico, privacidade de dados e transparência são desafios importantes...
[LOG] Chunk 2 (score: 0.4775): A inteligência artificial é um campo da ciência da computação que visa criar sistemas capazes de exe...
[LOG] Chunk 3 (score: 0.4212): Aprendizado por reforço é uma área da IA onde agentes aprendem a tomar decisões através de interação...
[LOG] Enviando prompt para Groq (modelo: llama-3.1-8b-instant)...
[LOG] Resposta recebida (primeiros 200 caracteres): As aplicações da IA mencionadas incluem a visão, a fala e a tomada de decisão, como mencionado na "Introdução à Inteligência Artificial (2023)". Além disso, a IA é utilizada em robótica, jogos e si